# Evaluación con/sin RAG en Colab — genera `eval_resultados.json`

Corre la evaluación (4 artefactos, con vs sin RAG) con **qwen2.5:14b-instruct** (el modelo que
usamos) y te **descarga** `data/eval_resultados.json` + la figura `figs/aporte_rag.png`.

**Antes de ejecutar:**
1. 🖥️ *Entorno de ejecución → Cambiar tipo de entorno → **T4 GPU***.
2. ▶️ Ejecutá la celda de abajo. Cuando lo pida, subí **`Entrega-Grupo14-GeneradorAcademico.zip`**.
3. ⏳ Paciencia: con 14b son **~8–12 min** (baja el modelo + evalúa). Al terminar se descargan los 2 archivos.

> Para una corrida más rápida (menor calidad) cambiá `OLLAMA_MODEL` a `qwen2.5:7b-instruct`.


In [ ]:
import os, sys, time, zipfile, subprocess
from google.colab import files

# 1) Subí el zip de la entrega
up = files.upload()
z = [n for n in up if n.endswith(".zip")][0]
with zipfile.ZipFile(z) as f:
    f.extractall(); root = f.namelist()[0].split("/")[0]
os.chdir(root)

# 2) Dependencias del proyecto
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])

# 3) Ollama + modelo en la GPU (14b = el que usamos de verdad)
subprocess.run("apt-get -qq install -y zstd pciutils", shell=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_MODEL"] = "qwen2.5:14b-instruct"   # <- 7b-instruct para ir más rápido
subprocess.Popen(["ollama", "serve"]); time.sleep(5)
subprocess.run(["ollama", "pull", os.environ["OLLAMA_MODEL"]])

# 4) Construir las bases y correr la evaluación (genera data/eval_resultados.json)
sys.path.insert(0, "src")
import db, ingest; db.reindexar(); ingest.reindexar()
subprocess.run([sys.executable, "scripts/run_eval.py"])
import figuras; figuras.fig_aporte_rag()   # figura con/sin RAG

# 5) Descargar resultados + figura
files.download("data/eval_resultados.json")
files.download("figs/aporte_rag.png")
